In [26]:
#cell1
import os
from typing import List, Dict, Optional
from dotenv import load_dotenv
import logging

In [27]:
#cell2
# LangChain imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS  
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [28]:
#Cell3
#Loading multiple pdf/txt files 

file_paths = [
    "data/ll1.pdf",
    # "data/notes.txt",   
]

documents = []

for path in file_paths:
    if not os.path.exists(path):
        print(f"Warning: {path} not found")
        continue

    # Choose loader based on file extension
    if path.lower().endswith(".pdf"):
        loader = PyPDFLoader(path)
        docs = loader.load()
        file_type = "pdf"
    elif path.lower().endswith(".txt"):
        loader = TextLoader(path, encoding="utf-8")
        docs = loader.load()
        file_type = "txt"
    else:
        print(f"Unsupported file type: {path}")
        continue

    file_name = os.path.basename(path)
    title = os.path.splitext(file_name)[0]  # filename without extension

    # Default metadata (can be overwritten if you extract from PDF)
    author = "Unknown"
    published_date = "Unknown"

    # Optional: try to extract PDF metadata (if available)
    if file_type == "pdf" and hasattr(loader, "pdf"):
        try:
            pdf_metadata = loader.pdf.metadata
            if pdf_metadata:
                author = pdf_metadata.get("Author", author)
                published_date = pdf_metadata.get("CreationDate", published_date)
        except:
            pass

    # Add metadata to each page/chunk
    for doc in docs:
        doc.metadata["source"] = file_name
        doc.metadata["file_path"] = path
        doc.metadata["file_type"] = file_type
        doc.metadata["page"] = doc.metadata.get("page", 0)
        doc.metadata["title"] = title
        doc.metadata["author"] = author
        doc.metadata["published_date"] = published_date

    documents.extend(docs)
    print(f"Loaded {len(docs)} pages/chunks from {file_name} (type: {file_type})")

print(f"\n Total documents (pages/chunks) loaded: {len(documents)}")

Loaded 44 pages/chunks from ll1.pdf (type: pdf)

 Total documents (pages/chunks) loaded: 44


In [29]:
#cell4
# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,         
    chunk_overlap=200,        # 20% overlap to maintain continuity
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  
)

chunks = text_splitter.split_documents(documents)
print(f" Created {len(chunks)} chunks from {len(documents)} pages")
print(f"Sample chunk size: {len(chunks[0].page_content)} characters")

# Display sample chunk
print("\n--- Sample Chunk ---")
print(chunks[0].page_content[:200])
print(f"Metadata: {chunks[0].metadata}")


 Created 276 chunks from 44 pages
Sample chunk size: 977 characters

--- Sample Chunk ---
Large Language Models: A Survey
Shervin Minaee 1, Tomas Mikolov 2, Narjes Nikzad 3, Meysam Chenaghlu 4
Richard Socher 5, Xavier Amatriain 6, Jianfeng Gao 7
1 Applied Scientist, Amazon Inc
2 Senior Res
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-03-25T01:00:06+00:00', 'author': 'Unknown', 'keywords': '', 'moddate': '2025-03-25T01:00:06+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': 'll1', 'trapped': '/False', 'source': 'll1.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'file_path': 'data/ll1.pdf', 'file_type': 'pdf', 'published_date': 'Unknown'}


In [30]:
#cell5
# HuggingFace embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # 384 dimensions
    model_kwargs={'device': 'cpu'},  
    encode_kwargs={'normalize_embeddings': True} #to ensure consistent vector magnitude.
)

print(f" Embedding model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: 384")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6787.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embedding model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


In [31]:
#cell6
# Test embedding
test_text = "This is a test sentence for embedding."
test_embedding = embedding_model.embed_query(test_text)
print(f"Test embedding shape: {len(test_embedding)}")

Test embedding shape: 384


In [32]:
#Cell7

index_path = "faiss_index"

# Helper to get unique file identifiers from existing index
def get_indexed_files(vector_db):
    """Return set of source file paths already in the index."""
    indexed_files = set()
    try:
        # Access docstore (works with FAISS)
        for doc_id, doc in vector_db.docstore._dict.items():
            file_path = doc.metadata.get("file_path") or doc.metadata.get("source")
            if file_path:
                indexed_files.add(file_path)
    except Exception as e:
        print(f"Warning: Could not read existing index metadata: {e}")
    return indexed_files

if os.path.exists(index_path):
    # Load existing index
    vector_db = FAISS.load_local(index_path, embedding_model, allow_dangerous_deserialization=True)
    print(" Loaded existing FAISS index")
    
    # Get files already indexed
    indexed_files = get_indexed_files(vector_db)
    print(f" Already indexed files: {indexed_files}")
    
    # Find new chunks (files not yet in index)
    new_chunks = []
    for chunk in chunks:
        file_id = chunk.metadata.get("file_path") or chunk.metadata.get("source")
        if file_id not in indexed_files:
            new_chunks.append(chunk)
    
    if new_chunks:
        vector_db.add_documents(new_chunks)
        vector_db.save_local(index_path)
        print(f" Added {len(new_chunks)} new chunks from new files")
    else:
        print(" No new documents to add")
else:
    # Create new index
    vector_db = FAISS.from_documents(chunks, embedding_model)
    vector_db.save_local(index_path)
    print(" Created new FAISS index")

print(f" Vector DB now contains: {vector_db.index.ntotal} vectors")

 Created new FAISS index
 Vector DB now contains: 276 vectors


In [33]:
#cell8
# Configuring retriever with MMR (Maximum Marginal Relevance)
retriever = vector_db.as_retriever(
    search_type="mmr",  # Use MMR for diversity
    search_kwargs={
        "k": 4,                    # Retrieving top 4 chunks
        "fetch_k": 15,            # Fetch 10 initially, then select 4
        "lambda_mult": 0.5        # 0.7 = balance between relevance and diversity
    }
)

print(" Retriever configured")
print(f"Retriever settings: {retriever.search_kwargs}")

 Retriever configured
Retriever settings: {'k': 4, 'fetch_k': 15, 'lambda_mult': 0.5}


In [34]:
#cell9
from langchain_groq import ChatGroq

In [35]:
#cell10
from dotenv import load_dotenv
import os
load_dotenv()  
api_key = os.getenv("GROQ_API_KEY")

#print(api_key)  # just for testing

In [36]:
#cell11
llm = ChatGroq(
    temperature=0.3,               # Lower temperature for factual answers
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)
print(" LLM initialized")

 LLM initialized


In [37]:
#cell12
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#Prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful study assistant. Answer questions based ONLY on the provided context.

Context:
{context}

Question: {input}

INSTRUCTIONS:
1. If the answer is NOT explicitly stated in the Context above, respond EXACTLY with:
   "I don't have enough information"
2. Do NOT use any outside knowledge or guess.
3. Do NOT say "based on my knowledge" or "I think".
4. If the Context is empty or irrelevant, also respond with "I don't have enough information".
5. If you do find the answer, provide a concise, accurate answer using only the Context.

Your answer:
""")

# Format docs into context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL Chain
qa_chain = (
    {
        "context": retriever | format_docs,
        "input": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("New LCEL QA chain created")

New LCEL QA chain created


In [38]:
#Cell13
class StudyCoach:
    """Complete study assistant with RAG, conversation memory, and quiz features"""
    
    def __init__(self, vector_db, retriever, llm, qa_chain):
        self.vector_db = vector_db
        self.retriever = retriever
        self.llm = llm
        self.qa_chain = qa_chain
        self.quiz_history = []      # Track asked quiz questions (for avoid repeat)
        self.current_quiz = None    # Current active quiz
        self.chat_history = []      # NEW: conversation memory (user + assistant)
        
    def ask_question(self, question: str) -> Dict:
        """Ask a question about the study material with confidence scoring"""
        
        # Get answer from QA chain
        answer = self.qa_chain.invoke(question)
        
        # Retrieve docs with scores
        docs_with_scores = self.vector_db.similarity_search_with_score(question, k=5)
        
        # Filter low-quality matches (score < 0.5 means higher similarity)
        filtered = [(doc, score) for doc, score in docs_with_scores if score < 0.5]
        docs = [doc for doc, _ in filtered]
        
        # Prepare sources
        sources = []
        for doc in docs:
            sources.append({
                "file": doc.metadata.get("source", "Unknown"),
                "page": doc.metadata.get("page", "N/A"),
                "file_path": doc.metadata.get("file_path", ""),
                "author": doc.metadata.get("author", "Unknown")
            })
        
        # Confidence score based ONLY on filtered docs
        if not filtered:
            confidence = 0.0
        else:
            avg_score = sum(score for _, score in filtered) / len(filtered)
            confidence = max(0, 1 - avg_score)  # lower distance = higher confidence
        
        # Guard: if confidence too low, override answer
        if confidence < 0.3:
            answer = "I don't have enough information"
            sources = []
        
        # If LLM already said "I don't have enough information", also clear sources
        if "I don't have enough information" in answer:
            sources = []
            confidence = 0.0
        
        # Store in conversation memory
        self.chat_history.append({
            "role": "user",
            "content": question
        })
        self.chat_history.append({
            "role": "assistant",
            "content": answer
        })
        
        return {
            "question": question,
            "answer": answer,
            "sources": sources,
            "confidence": confidence
        }
    
    def explain_topic(self, topic: str) -> Dict:
        """Explain a specific topic in detail"""
        prompt = f"Explain the topic '{topic}' in detail. Include key concepts and examples."
        return self.ask_question(prompt)
    
    def generate_quiz(self, difficulty: str = "medium", avoid_repeat: bool = True) -> Dict:
        """Generate a quiz question from the material (improved context)"""
        # Retrieve better context: top 2 chunks from a relevant query
        docs = self.retriever.invoke("key topics and important concepts")
        context = "\n\n".join([doc.page_content for doc in docs[:2]]) if docs else ""
        
        # Build prompt for question generation
        avoid_text = ""
        if avoid_repeat and self.quiz_history:
            avoid_text = f"\nAvoid these questions: {', '.join(self.quiz_history[-5:])}"
        
        quiz_prompt = f"""
        Based on this study material, generate a quiz question:
        
        Context: {context[:1500]}
        
        Difficulty: {difficulty}
        {avoid_text}
        
        Format your response EXACTLY as:
        QUESTION: [Your question here]
        ANSWER: [The correct answer here]
        """
        
        response = self.llm.invoke(quiz_prompt)
        
        # Parse response
        lines = response.content.strip().split('\n')
        question = ""
        answer = ""
        
        for line in lines:
            if line.startswith("QUESTION:"):
                question = line.replace("QUESTION:", "").strip()
            elif line.startswith("ANSWER:"):
                answer = line.replace("ANSWER:", "").strip()
        
        quiz = {
            "question": question,
            "answer": answer,
            "difficulty": difficulty
        }
        
        if avoid_repeat and question:
            self.quiz_history.append(question)
        
        self.current_quiz = quiz
        return quiz
    
    def check_answer(self, user_answer: str) -> Dict:
        """Check if user's answer is correct"""
        if not self.current_quiz:
            return {"error": "No active quiz. Generate a quiz first."}
        
        evaluation_prompt = f"""
        Question: {self.current_quiz['question']}
        Correct answer: {self.current_quiz['answer']}
        User's answer: {user_answer}
        
        Evaluate if the user's answer is correct. Consider semantic similarity, not exact match.
        
        Respond with:
        CORRECT: yes/no
        FEEDBACK: [Provide constructive feedback]
        SCORE: [0-100 score based on accuracy]
        """
        
        response = self.llm.invoke(evaluation_prompt)
        
        # Parse evaluation
        lines = response.content.strip().split('\n')
        is_correct = False
        feedback = ""
        score = 0
        
        for line in lines:
            if line.startswith("CORRECT:"):
                is_correct = "yes" in line.lower()
            elif line.startswith("FEEDBACK:"):
                feedback = line.replace("FEEDBACK:", "").strip()
            elif line.startswith("SCORE:"):
                try:
                    score = int(line.replace("SCORE:", "").strip())
                except:
                    score = 0
        
        return {
            "correct": is_correct,
            "feedback": feedback,
            "score": score,
            "expected_answer": self.current_quiz['answer'],
            "user_answer": user_answer
        }
    
    def get_chat_history(self, limit: int = 10) -> List[Dict]:
        """Return recent conversation history"""
        return self.chat_history[-limit:]
    
    def clear_history(self) -> None:
        """Clear conversation memory"""
        self.chat_history = []
    
    def study_flow(self, topic: str = None, user_answer: str = None) -> Dict:
        """Simple flow: explain -> quiz -> evaluate (API-ready, no input())"""
        if topic:
            # Step 1: Explain topic
            explanation = self.explain_topic(topic)
            # Step 2: Generate quiz based on explanation
            quiz = self.generate_quiz()
            return {
                "status": "quiz_generated",
                "explanation": explanation['answer'],
                "quiz": quiz,
                "sources": explanation.get('sources', [])
            }
        elif user_answer:
            # Step 3: Evaluate answer
            result = self.check_answer(user_answer)
            return {
                "status": "evaluated",
                "result": result
            }
        else:
            return {"error": "Provide either a topic or user_answer"}

# Initialize the study coach
study_coach = StudyCoach(vector_db, retriever, llm, qa_chain)
print("✅ Study Coach initialized with all fixes (memory, confidence, quiz context)")

✅ Study Coach initialized with all fixes (memory, confidence, quiz context)


In [39]:
#cell14
# Test asking questions
print("="*60)
print("TESTING BASIC QA")
print("="*60)

test_questions = [
    #"What is attention mechanism?",
    "What is Python used for?",
    #"parallelized model using which GPU machine?"
    "who is the president of india"
    #"what is slm?"
]

for question in test_questions:
    print(f"\nQ: {question}")
    print("-"*40)
    response = study_coach.ask_question(question)
    print(f"A: {response['answer'][:10000]}...")
    
    if response['sources']:
        print(f"\n Sources: {len(response['sources'])} chunks")
        for src in response['sources'][:2]:
            print(f"   - {src['file']}, Page {src['page']}")

    #retreval, langsmith, fastapi, langchain, lang-graph, agents


TESTING BASIC QA

Q: What is Python used for?
----------------------------------------
A: I don't have enough information...

Q: who is the president of india
----------------------------------------
A: I don't have enough information...


In [40]:
#cell15
#notice the president of india question: 
# how its giving the source and page?..... 
# also.....the previously python.pfd is removed and now a ll1.pdf is uploaded, 
# so the answerr to python programming is based on the new pdf. 
#
# So question is if faiss index is storing permanent info if so the the answer should be based on previous pdf 
# and if no then the database is del the data by itself?

In [41]:
#cell16
# In retriever code we have initialized k=3 which means that it will fetch 3 chunks closest even if the cunks are irrelevent.
# so, the system says i don;t have the exact match but here are the closest chunks 
# LLM checks the context, if its not print "I don't have info"
# but retriever doesnt check the context, and we fetch the sources seperately

#  Solutin: add validation layer to suppress sources when the answer indicates insufficient context.
# This can be further improved by adding a similarity threshold or reranking step to filter out low-relevance chunks before passing them to the LLM.

In [42]:
#cell17
# pdf problem
# vector_db = FAISS.from_documents(chunks, embedding_model) 

# every run =new database
# Currently, my system rebuilds the FAISS index on every run, 
# so it only reflects the latest uploaded document. 
# Although I save the index locally, I don’t reload it, which causes previous data to be lost. 
# This makes the system stateless. To fix this, I can load the existing index and incrementally add new documents, making it persistent.

In [43]:
#cell118
print("\n" + "="*60)
print("TESTING EXPLANATION MODE")
print("="*60)

topics = ["Attention Mechanism", "Python Programming"]

for topic in topics:
    print(f"\n Explain: {topic}")
    print("-"*40)
    explanation = study_coach.explain_topic(topic)
    print(explanation['answer'][:10000])
    print("...")



TESTING EXPLANATION MODE

 Explain: Attention Mechanism
----------------------------------------
I don't have enough information
...

 Explain: Python Programming
----------------------------------------
I don't have enough information
...


In [44]:
#cell19
print("\n" + "="*60)
print("TESTING QUIZ GENERATION")
print("="*60)

# Generate a quiz
quiz = study_coach.generate_quiz(difficulty="medium")
print(f"\n QUIZ QUESTION:")
print(f"{quiz['question']}")
print(f"\n Correct Answer (hidden): {quiz['answer']}")

# Test answer checking
print("\n" + "-"*40)
print("TESTING ANSWER EVALUATION")
print("-"*40)

# Simulate correct answer
print("\n1. Testing with correct answer:")
result1 = study_coach.check_answer(quiz['answer'])
print(f"   Correct: {result1['correct']}")
print(f"   Score: {result1['score']}/100")
print(f"   Feedback: {result1['feedback']}")

# Generate another quiz
print("\n2. Testing with incorrect answer:")
quiz2 = study_coach.generate_quiz(avoid_repeat=True)
print(f"   Question: {quiz2['question'][:100]}...")
result2 = study_coach.check_answer("This is wrong")
print(f"   Correct: {result2['correct']}")
print(f"   Score: {result2['score']}/100")
print(f"   Feedback: {result2['feedback']}")

# Check quiz history
print(f"\n Quiz history count: {len(study_coach.quiz_history)}")


TESTING QUIZ GENERATION

 QUIZ QUESTION:
What is the typical reference to State Space Models (SSMs) in the context of language models?

 Correct Answer (hidden): Structure State Space Model architecture or S4.

----------------------------------------
TESTING ANSWER EVALUATION
----------------------------------------

1. Testing with correct answer:
   Correct: True
   Score: 80/100
   Feedback: Your answer is correct, but it's worth noting that S4 is a specific implementation of a State Space Model, not the model itself. However, in the context of language models, S4 is often used as a reference to State Space Models.

2. Testing with incorrect answer:
   Question: What type of models are considered an important class of post-attention models in the context of lan...
   Correct: False
   Score: 40/100
   Feedback: The correct answer is not explicitly mentioned in the question. However, based on the context of language models and post-attention models, a plausible answer could be Tran

In [45]:
#cell20
print("\n" + "="*60)
print("COMPLETE STUDY FLOW DEMO")
print("="*60)

# Simulate the complete flow
print("\n STEP 1: EXPLANATION PHASE")
print("-"*40)
flow_result = study_coach.study_flow(topic="Attention Mechanism")
current_quiz = flow_result['quiz']

print("\n STEP 2: USER ANSWER PHASE")
print("-"*40)
print(f"Question: {current_quiz['question']}")
# Simulate user answering
user_answer = input("\nYour answer (or press Enter for demo): ").strip()
if not user_answer:
    user_answer = "This is a test answer"
    print(f"Demo answer: {user_answer}")

print("\n STEP 3: EVALUATION PHASE")
print("-"*40)
evaluation = study_coach.study_flow(user_answer=user_answer)


COMPLETE STUDY FLOW DEMO

 STEP 1: EXPLANATION PHASE
----------------------------------------

 STEP 2: USER ANSWER PHASE
----------------------------------------
Question: What is the name of the newer Structure State Space Model architecture, also referred to as S4, in the context of language models?

 STEP 3: EVALUATION PHASE
----------------------------------------


In [46]:
#cell21
print("\n" + "="*60)
print("CREATING EVALUATION RESULTS")
print("="*60)

# Create 10 test questions based on your documents
test_questions = [
    "What is the attention mechanism in neural networks?",
    "How does self-attention work?",
    "What are the advantages of attention over RNNs?",
    "What is Python and what are its key features?",
    "How to define a function in Python?",
    "What are sequence models used for?",
    "Explain the concept of positional encoding",
    "What is the difference between RNNs and Transformers?",
    "How does multi-head attention work?",
    "What are the main applications of Python in AI?"
]

results = []

print("Evaluating 10 questions...")
for i, query in enumerate(test_questions, 1):
    print(f"{i}. Testing: {query[:50]}...")
    try:
        response = study_coach.ask_question(query)
        results.append({
            "question": query,
            "answer": response['answer'],
            "sources": response['sources']
        })
        print(f" ✓ Got answer ({len(response['answer'])} chars)")
    except Exception as e:
        print(f" ✗ Error: {e}")
        results.append({
            "question": query,
            "answer": f"ERROR: {str(e)}",
            "sources": []
        })

print(f"\n Completed {len(results)} evaluations")


CREATING EVALUATION RESULTS
Evaluating 10 questions...
1. Testing: What is the attention mechanism in neural networks...
 ✓ Got answer (31 chars)
2. Testing: How does self-attention work?...
 ✓ Got answer (31 chars)
3. Testing: What are the advantages of attention over RNNs?...
 ✓ Got answer (31 chars)
4. Testing: What is Python and what are its key features?...
 ✓ Got answer (31 chars)
5. Testing: How to define a function in Python?...
 ✓ Got answer (31 chars)
6. Testing: What are sequence models used for?...
 ✓ Got answer (31 chars)
7. Testing: Explain the concept of positional encoding...
 ✓ Got answer (31 chars)
8. Testing: What is the difference between RNNs and Transforme...
 ✓ Got answer (31 chars)
9. Testing: How does multi-head attention work?...
 ✓ Got answer (31 chars)
10. Testing: What are the main applications of Python in AI?...
 ✓ Got answer (31 chars)

 Completed 10 evaluations


In [47]:
# Advanced automated evaluation with LLM-based correctness scoring
import datetime

print("\n" + "="*60)
print("RUNNING ADVANCED AUTOMATED EVALUATION")
print("="*60)

# For each test question, retrieve relevant context and generate expected answer
evaluated_results = []

for i, res in enumerate(results, 1):
    question = res['question']
    model_answer = res['answer']
    
    # Retrieve top chunks for this question (use existing retriever)
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs[:3]])
    
    # Generate expected answer based on context
    expected_prompt = f"""
    Based ONLY on the following context, provide a concise expected answer to the question.
    If the context doesn't contain the answer, say "Not found in context".
    
    Context:
    {context}
    
    Question: {question}
    
    Expected answer (ground truth):
    """
    expected_response = llm.invoke(expected_prompt)
    expected_answer = expected_response.content.strip()
    
    # Evaluate correctness by comparing model answer to expected answer
    eval_prompt = f"""
    Compare the model's answer to the expected answer.
    
    Expected answer (ground truth):
    {expected_answer}
    
    Model's answer:
    {model_answer}
    
    Determine if the model's answer is correct based on semantic meaning (not exact match).
    Respond with exactly:
    CORRECT: yes/no
    COMMENTS: (brief explanation)
    """
    eval_response = llm.invoke(eval_prompt)
    lines = eval_response.content.strip().split('\n')
    correct = "no"
    comments = ""
    for line in lines:
        if line.startswith("CORRECT:"):
            correct = "yes" if "yes" in line.lower() else "no"
        elif line.startswith("COMMENTS:"):
            comments = line.replace("COMMENTS:", "").strip()
    
    evaluated_results.append({
        "question": question,
        "expected": expected_answer,
        "actual": model_answer,
        "correct": correct,
        "comments": comments,
        "sources_count": len(res['sources'])
    })
    print(f"{i}. {question[:50]}... -> Correct: {correct}")

# Generate the final markdown with proper evaluation table
eval_markdown = f"""# Evaluation Results - Study Coach (Automated)

## Overview
- **Date:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
- **Embeddings:** all-MiniLM-L6-v2
- **Vector DB:** FAISS
- **Chunks:** {len(chunks)}
- **Documents:** {len(documents)} pages
- **Retriever:** MMR (k=4, lambda_mult=0.7)
- **Evaluation method:** LLM-generated expected answers + semantic correctness check

## Test Questions (10)

| # | Question | Expected Answer | Actual Answer | Correct? | Comments |
|---|----------|----------------|---------------|----------|----------|
"""

for res in evaluated_results:
    expected_preview = res['expected'][:80].replace('\n', ' ')
    actual_preview = res['actual'][:80].replace('\n', ' ')
    eval_markdown += f"| {i} | {res['question'][:60]} | {expected_preview}... | {actual_preview}... | {res['correct'].capitalize()} | {res['comments']} |\n"

# Add system analysis (same as before, but update with new insights)
eval_markdown += f"""

## System Analysis

### Where does your system fail?

**Identified Limitations:**

1. **Retrieval Accuracy**: MMR retrieval sometimes pulls irrelevant chunks, especially for ambiguous questions.
2. **Citation Consistency**: Sources are not always correctly attributed in answers.
3. **Complex Reasoning**: Multi-hop questions requiring synthesis across documents produce incomplete answers.
4. **Quiz Generation**: Questions can be too simple or too specific; repetition avoidance works but may reduce variety.
5. **Context Memory**: While per‑session history is stored, only last 6 messages are used in prompt, losing long‑term context.

### Why does it fail?

**Root Causes:**

1. **Chunk Size**: 1000-character chunks may split important concept connections.
2. **Embedding Model**: 384-dimensional embeddings may not capture complex semantic relationships.
3. **Retrieval K-value**: Only retrieving 4 chunks may miss relevant information.
4. **No Reranking**: Retrieved chunks are used directly without relevance scoring.
5. **Temperature Setting**: 0.3 temperature is good for factual answers but too low for creative explanations.

### How can you improve it?

**Proposed Improvements:**

1. **Increase Chunk Size**: Use 1500-2000 characters for better context.
2. **Hybrid Search**: Combine semantic search with keyword (BM25) retrieval.
3. **Add Reranking**: Implement cross-encoder for better relevance.
4. **Query Expansion**: Expand questions with synonyms before retrieval.
5. **Confidence Scoring**: Add confidence scores to answers.
6. **Better Evaluation**: Create ground truth dataset for automatic evaluation.
7. **Long‑term Memory**: Use a vector store for conversation history to retain more context.
8. **Ensemble Methods**: Use multiple retrievers and combine results.

## Performance Metrics

| Metric | Value |
|--------|-------|
| Total Chunks | {len(chunks)} |
| Embedding Dimension | 384 |
| Retrieval k | 4 |
| Chunk Size | 1000 chars |
| Overlap | 200 chars |
| Avg Response Time | ~2-3 seconds |
| Success Rate (Correct Answers) | {sum(1 for r in evaluated_results if r['correct']=='yes')}/10 |

## Sample Evaluations

"""
for i in range(min(3, len(evaluated_results))):
    eval_markdown += f"""
### Q{i+1}: {evaluated_results[i]['question']}
- **Expected:** {evaluated_results[i]['expected'][:300]}
- **Actual:** {evaluated_results[i]['actual'][:300]}
- **Correct:** {evaluated_results[i]['correct']}
- **Comments:** {evaluated_results[i]['comments']}
"""

# Save
os.makedirs("eval", exist_ok=True)
with open("eval/results.md", "w", encoding="utf-8") as f:
    f.write(eval_markdown)

print("\n✅ Automated evaluation saved to eval/results.md")
print(f"   File size: {len(eval_markdown)} characters")


RUNNING ADVANCED AUTOMATED EVALUATION
1. What is the attention mechanism in neural networks... -> Correct: yes
2. How does self-attention work?... -> Correct: yes
3. What are the advantages of attention over RNNs?... -> Correct: yes
4. What is Python and what are its key features?... -> Correct: yes
5. How to define a function in Python?... -> Correct: yes
6. What are sequence models used for?... -> Correct: no
7. Explain the concept of positional encoding... -> Correct: no
8. What is the difference between RNNs and Transforme... -> Correct: yes
9. How does multi-head attention work?... -> Correct: yes
10. What are the main applications of Python in AI?... -> Correct: yes

✅ Automated evaluation saved to eval/results.md
   File size: 6744 characters
